In [ ]:
from pybliometrics.scopus import AuthorSearch
from pybliometrics.scopus import AuthorRetrieval
from scholarly import scholarly
import pandas as pd
import sys
sys.path.append("..")
from rcn_py import orcid, data_storage, neo4j_rsd, openalex



In [ ]:
fullname = "Peter Kalverla"
orcid_id = "0000-0002-5025-7862"

author_last = "Kalverla"
author_first = "Peter"

### Scopus

In [ ]:
s = AuthorSearch('AUTHLAST('+author_last+') and AUTHFIRST('+author_first+')')
author_id = s.authors[0].eid.split('-')[-1]

In [ ]:
au = AuthorRetrieval(author_id)
docs = pd.DataFrame(au.get_documents())

### Google Scholar

In [ ]:
# search publication info using title
search_pub = scholarly.search_pubs('17B. 6 LONG TERM OBSERVATIONS OF LONG WAVE RADIATIVE FLUX DIVERGENCE IN THE STABLE BOUNDARY LAYER OVER LAND')
pubinfo=next(search_pub)

In [ ]:
# author info searched by scholar id
search_query_author = scholarly.search_author_id("TOokPzcAAAAJ")
search_query_author

In [ ]:
# author info using name
search_query_author2 = scholarly.search_author('Peter Kalverla')
authorinfo = scholarly.fill(next(search_query_author2))

In [ ]:
# publication info
data_storage.scholar_author_info(fullname)

### Crossref (ORCID)

In [ ]:
# Using rcn_py.orcid functions
orcid_id = orcid.name_to_orcid_id(fullname)
orcid_record = orcid.query_orcid_for_record(orcid_id)
docs = orcid.extract_works_section(orcid_record)

In [ ]:
docs[0]

In [ ]:
# All publication info of one researcher
data_storage.crossref_author_by_name(fullname)

In [ ]:
data_storage.crossref_author_by_orcid(orcid)

In [ ]:
# Get all publications of all coauthors and do de-duplication
# Long running time
df = data_storage.coauthor_data_crossref(orcid_id)

In [ ]:
node_data = data_storage.assign_group_node(orcid_id)

In [ ]:
links = data_storage.get_links_from_coauthor_rel(orcid_id)

In [ ]:
data_storage.build_networkx(orcid_id)

In [ ]:
data_storage.build_networkx('0000-0002-5025-7862')

### RSD

In [ ]:
rsd_data = neo4j_rsd.request_rsd_data()

### OpenAlex

In [ ]:
author_result_1 = openalex.search_user_by_orcid(orcid_id)
author_result_2 = openalex.search_user_by_name(fullname)


In [ ]:
works_1 = openalex.search_works_by_orcid(orcid_id)
works_2 = openalex.search_works_by_name(fullname)


In [ ]:
doi = "https://doi.org/10.7717/peerj.4375"
publication_result = openalex.get_pub_info_by_doi(doi)      # all the information of this publication
keywords = openalex.publication_keywords(doi)               # for topic modeling

In [ ]:
institution_name = "Netherlands eScience Center"
institution = openalex.find_institution(institution_name)

inst_works_1 = openalex.get_some_works_of_one_institution(institution_name)     # Fast, only return 25 work results
inst_works_2 = openalex.get_works_of_one_institution(institution_name, 100)     # Set work count
inst_works_3 = openalex.get_all_works_of_one_institution(institution_name)      # All works
inst_works_4 = openalex.get_works_of_one_institution_by_year(institution_name, 2021, 2023) # Year period 

In [ ]:
colla_institutions = openalex.coauthor_institutions(institution_name)               # Collaboration between institutions
node_list, coauthor_pair = openalex.build_institution_network(institution_name)     # Create nodes and relations of the network